# **Machine Learning Basics**

---



## ***RAG Implementation***

-------



In [15]:
# 📦 INSTALLS
!pip install -q openai chromadb pypdf tiktoken ipywidgets

exit

In [16]:
# 📚 IMPORTS
import uuid
import textwrap
#----------------------------------------
from google.colab import files
from google.colab import userdata
#----------------------------------------
from openai import OpenAI
from pypdf import PdfReader
#----------------------------------------
import chromadb

# 🔑 OPENAI CLIENT
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI( api_key= OPENAI_API_KEY)

# 📄 LOAD PDF
def load_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            text += extracted + "\n"
    return text

# ✂️ CHUNKER WITH OVERLAP
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        # move forward with overlap
        start += chunk_size - overlap
    return chunks

# 🧠 CREATE EMBEDDING
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# 🗂️ CHROMADB SETUP
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="simple_rag"
)

# 📥 STORE CHUNKS
def add_chunks_to_chroma(chunks):
    for chunk in chunks:
        embedding = get_embedding(chunk)
        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding]
        )

    print(f"Stored {len(chunks)} chunks in ChromaDB.")


# 🔎 RETRIEVE DOCUMENTS
def retrieve(query, top_k=3):
    query_embedding = get_embedding(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )
    docs = results["documents"][0]
    return docs

# 🤖 ASK QUESTION
def ask(query):
    docs = retrieve(query)
    context = "\n\n".join(docs)
    prompt = f"""
    You are a helpful RAG assistant.
    Answer ONLY from the provided context.
    If the answer is not in the context, say:
    "I could not find the answer in the document."
    Context:
    {context}
    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You answer questions using retrieved documents."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# 📂 UPLOAD PDF
print("Upload your PDF file.\n")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"\nLoaded PDF: {pdf_path}")

# 🏗️ BUILD RAG PIPELINE
print("\nLoading PDF...")
text = load_pdf(pdf_path)
print("PDF loaded.")
print("\nChunking document...")
chunks = chunk_text(text, chunk_size=800, overlap=200)
print(f"Created {len(chunks)} chunks.")
print("\nGenerating embeddings and storing in ChromaDB...")
add_chunks_to_chroma(chunks)
print("\n✅ RAG Pipeline Ready")

# 💬 INTERACTIVE CHATBOT
print("\n🤖 Chatbot Ready")
print("Type 'exit' to stop.\n")
while True:
    query = input("You: ")
    if query.lower() == "exit":
        print("\nBot: Goodbye.")
        break
    answer = ask(query)
    print("\nBot:")
    print(textwrap.fill(answer, width=100))
    print("\n" + "=" * 100 + "\n")

Upload your PDF file.



KeyboardInterrupt: 

## ***Embeddings for Sentances***

In [11]:
!pip install -q sentence-transformers

In [12]:
from sentence_transformers import SentenceTransformer
sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[[ 6.76569194e-02  6.34959713e-02  4.87130992e-02  7.93050006e-02
   3.74480896e-02  2.65282183e-03  3.93749885e-02 -7.09845545e-03
   5.93613982e-02  3.15369926e-02  6.00981042e-02 -5.29052094e-02
   4.06067520e-02 -2.59308629e-02  2.98427939e-02  1.12692360e-03
   7.35148415e-02 -5.03819063e-02 -1.22386619e-01  2.37028319e-02
   2.97265295e-02  4.24768366e-02  2.56337635e-02  1.99515908e-03
  -5.69190197e-02 -2.71597710e-02 -3.29035483e-02  6.60248995e-02
   1.19007207e-01 -4.58790772e-02 -7.26214647e-02 -3.25840227e-02
   5.23413233e-02  4.50553633e-02  8.25300440e-03  3.67024057e-02
  -1.39415190e-02  6.53918087e-02 -2.64272243e-02  2.06387354e-04
  -1.36643304e-02 -3.62810828e-02 -1.95043813e-02 -2.89738160e-02
   3.94270197e-02 -8.84090587e-02  2.62429030e-03  1.36713982e-02
   4.83062863e-02 -3.11566275e-02 -1.17329143e-01 -5.11690117e-02
  -8.85287672e-02 -2.18962654e-02  1.42986597e-02  4.44167443e-02
  -1.34815564e-02  7.43392110e-02  2.66382731e-02 -1.98762864e-02
   1.79191